# AIXL distilled translator — fine-tune en Colab (GPU T4 gratis)

Entrena un modelo pequeño (Qwen2.5-3B-Instruct) con LoRA para que aprenda a codificar texto en AIXL 0.3 **sin necesitar la tarjeta de 19.500 caracteres en el prompt** — las reglas quedan guardadas en los pesos del modelo. El resultado se exporta a GGUF para servirlo localmente con Ollama en el Mac.

**Antes de correr:** En el menú `Entorno de ejecución > Cambiar tipo de entorno de ejecución`, elige GPU (T4 es suficiente y gratis).

**Paso manual:** cuando la celda de carga de datos lo pida, sube el archivo `distill/corpus.jsonl` de este repo (1417 pares texto→AIXL ya validados, generados sin costo nuevo a partir de experimentos previos del proyecto).

In [ ]:
!pip install -q unsloth trl peft accelerate bitsandbytes

In [ ]:
from google.colab import files
print("Sube distill/corpus.jsonl")
uploaded = files.upload()
corpus_path = list(uploaded.keys())[0]

In [ ]:
import json

pairs = [json.loads(l) for l in open(corpus_path, encoding="utf-8") if l.strip()]
print("pares cargados:", len(pairs))
print(pairs[0])

# split: 90% train, 10% validation held out during training (NOT the same as blind5 — this is just
# to watch for overfitting during the run; the real honest accuracy number comes later from blind5
# via benchmarks/llm_translator_eval.py, run locally on the Mac after the model is back in Ollama)
import random
random.seed(0)
random.shuffle(pairs)
n_val = max(1, len(pairs) // 10)
val_pairs, train_pairs = pairs[:n_val], pairs[n_val:]
print("train:", len(train_pairs), "val:", len(val_pairs))

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512  # short: no giant card in the prompt, text + one AIXL line

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=0,
)

In [ ]:
# Short, fixed instruction — the whole point of fine-tuning is that the model no longer needs
# card_0.3.md (19.5k chars) at inference time; the encoding rules live in the weights instead.
SYSTEM_PROMPT = (
    "You are an AIXL 0.3 encoder. Given an instruction in Spanish, English or Portuguese, output "
    "its AIXL encoding as exactly one line starting with V:AIXL-0.3. Output only that line, nothing else."
)

def to_chat_text(example):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["text"]},
        {"role": "assistant", "content": example["aixl"]},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

from datasets import Dataset

train_ds = Dataset.from_list([{"text": to_chat_text(p)} for p in train_pairs])
val_ds = Dataset.from_list([{"text": to_chat_text(p)} for p in val_pairs])
print(train_ds[0]["text"])

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=SFTConfig(
        per_device_train_batch_size=16,
        gradient_accumulation_steps=2,
        warmup_steps=10,
        num_train_epochs=8,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=0,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer.train()

In [ ]:
# quick sanity check before spending time on GGUF export
FastLanguageModel.for_inference(model)
test_texts = ["Analiza las ventas de Q1 2026.", "Delete the report if it is older than 30 days."]
for t in test_texts:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": t}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=80, use_cache=True)
    print(t, "->", tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# export merged model to GGUF (q4_k_m — good balance for a 1.5B model on an 8GB Mac) for Ollama
model.save_pretrained_gguf("aixl-qwen3b-distilled", tokenizer, quantization_method="q4_k_m")

In [ ]:
import glob, hashlib
gguf_files = glob.glob("aixl-qwen3b-distilled_gguf/*.gguf")
print(gguf_files)
h = hashlib.sha256(open(gguf_files[0], "rb").read()).hexdigest()
print("SHA256:", h)
files.download(gguf_files[0])

## Siguiente paso (en el Mac, no en Colab)

1. Guarda el `.gguf` descargado en `~/.claude/skills/aixl-core/distill/`.
2. Crea un `Modelfile` junto a él:
   ```
   FROM ./aixl-qwen3b-distilled.gguf
   SYSTEM """You are an AIXL 0.3 encoder. Given an instruction in Spanish, English or Portuguese, output its AIXL encoding as exactly one line starting with V:AIXL-0.3. Output only that line, nothing else."""
   ```
3. `ollama create aixl-distilled -f Modelfile`
4. Medir F1 real contra `data/llm_translator/texts_blind5.json` (el set que quedó limpio, nunca usado como entrenamiento) con la misma metodología de `benchmarks/llm_translator_eval.py` usada para el modelo base.